# PersonaMem Experiment: Persistent Mem0 + QA

This notebook runs a **single experiment** with:

- **Persistent on-disk memory** scoped by `experiment_id` (memory survives restarts and is isolated per experiment).
- **Data separation**: stored `user_id` is prefixed with `experiment_id` (e.g. `text_gpt-4.1.-mini_graph:123`) so records are not mixed across experiments.
- **Same config for insertion and retrieval**: graph (Neo4j) is used when possible; after **3 retries** on failure we fall back to vector-only.
- **Structured logs** under `benchmark_logs/<experiment_id>/` for filling (errors, tokens, latency) and QA (retrieved memories, Mem0 answer, correct answer, question metadata).

**Chapters:**

1. **Chapter 1: Filling the memory** — Ingest PersonaMem user bundles into Mem0.
2. **Chapter 2: Running the QA** — Run benchmark questions against memory and log results.
3. **Chapter 3: LLM-as-a-Judge** — (To be added later.)

## Setup

Define the **experiment ID** (used for persistent memory path and log folder) and paths to the PersonaMem bundles JSONL.

**Important:** Run this notebook from the **project root** (the `PersonaMem0` directory) so that `Path.cwd()` and relative paths (e.g. `exports/`, `benchmark_logs/`) resolve correctly.

In [1]:
from pathlib import Path

# Experiment ID: memory storage and logs are scoped to this ID.
# Memory is stored under ~/.mem0/experiments/<EXPERIMENT_ID>/qdrant (persistent, on-disk).
# Logs are written to benchmark_logs/<EXPERIMENT_ID>/.
EXPERIMENT_ID = "text_gpt-4.1.-mini_graph"

# Path to PersonaMem user bundles (chat history + benchmark rows).
# Use the text 32k export for this experiment. Requires running from project root.
PROJECT_ROOT = Path.cwd()
JSONL_PATH = PROJECT_ROOT / "exports" / "personamem_benchmark_text_32k_user_bundles.jsonl"

# Optional: cap users/QA for quick runs (set to None for full run).
MAX_USERS = None
MAX_QA_PER_USER = None

assert JSONL_PATH.is_file(), f"JSONL not found: {JSONL_PATH}"
print(f"Experiment ID: {EXPERIMENT_ID}")
print(f"Bundles: {JSONL_PATH}")
print(f"Log dir: benchmark_logs/{EXPERIMENT_ID}/")

Experiment ID: text_gpt-4.1.-mini_graph
Bundles: /Users/viktorzozula/MemoriesTesting/PersonaMem0/exports/personamem_benchmark_text_32k_user_bundles.jsonl
Log dir: benchmark_logs/text_gpt-4.1.-mini_graph/


---
# Chapter 1: Filling the memory in

Load PersonaMem user bundles and add each user's chat history to Mem0. Memory is **persistent** and scoped to `EXPERIMENT_ID`. In storage, each user is stored as `EXPERIMENT_ID:user_id` so data is separated per experiment. Graph is used when possible; after 3 retries on failure we fall back to vector-only. If the LLM returns truncated or malformed JSON, a **safe-parse repair** (`mem0_safe_json`) is applied so partial memories can still be stored.

**Logs** (in `benchmark_logs/<EXPERIMENT_ID>/`):

- `stage1_fill_<timestamp>.json` — Full log: per-user wall time, input tokens, latency, success/error, add_calls, used_graph, graph_error, and config.
- `stage1_errors_<timestamp>.jsonl` — One JSON object per error line for easy parsing.

In [ ]:
from experiment_runner import (
    run_stage1_fill_experiment,
    get_experiment_log_dir,
)

# Use EXPERIMENT_ID from Setup; graph retries = 3 then fallback to no-graph.
log_dir = get_experiment_log_dir(EXPERIMENT_ID)
print(f"Log directory: {log_dir}")

log1 = run_stage1_fill_experiment(
    experiment_id=EXPERIMENT_ID,
    jsonl_path=JSONL_PATH,
    split="benchmark_text",
    max_users=MAX_USERS,
    use_async=False,
    max_concurrent=5,
    graph_retries=3,
)

print(f"Stage 1 complete: {log1.num_users} users, {log1.total_add_calls} successful adds, {len(log1.errors)} errors")
print(f"Total wall time: {log1.total_wall_seconds:.1f}s")
if log1.total_input_tokens:
    print(f"Total input tokens (approx): {log1.total_input_tokens}")
# Whether graph (Neo4j) was used for this run (same for all users)
used_graph_s1 = log1.per_user[0].get("used_graph") if log1.per_user else None
print(f"Graph memory: {'enabled' if used_graph_s1 else 'disabled (vector-only fallback)'}")

Log directory: /Users/viktorzozula/MemoriesTesting/PersonaMem0/benchmark_logs/text_gpt-4.1.-mini_graph


/Users/viktorzozula/MemoriesTesting/PersonaMem0/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
No sentence-transformers model found with name cross-encoder/ms-marco-MiniLM-L-6-v2. Creating a new one with mean pooling.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1609.20it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
classifier.bias              | UNEXPECTED |  | 
bert.embeddings.position_ids | UNEXPECTED |  | 
classifier.weight            | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Stage 1 (fill memory):  

In [3]:
# Inspect last few per-user records (wall_seconds, input_tokens, success, used_graph)
import json
for rec in log1.per_user[-5:]:
    print(json.dumps({k: rec.get(k) for k in ["user_id", "wall_seconds", "input_tokens", "success", "used_graph", "error"] if rec.get(k) is not None}, indent=2))

{
  "user_id": 10,
  "wall_seconds": 57.84730695901089,
  "input_tokens": 31516,
  "success": true,
  "used_graph": true
}
{
  "user_id": 23,
  "wall_seconds": 155.46409633298754,
  "input_tokens": 31948,
  "success": true,
  "used_graph": true
}
{
  "user_id": 25,
  "wall_seconds": 46.45655737500056,
  "input_tokens": 31759,
  "success": true,
  "used_graph": true
}


---
# Chapter 2: Running the QA

For each benchmark question we **search** memory (with graph when possible; 3 retries then fallback), then generate an **answer** with the same LLM (e.g. gpt-4.1-mini). Search uses experiment-scoped user IDs (`EXPERIMENT_ID:user_id`) so only this experiment's memories are queried. Logs include Mem0 retrieved memories, Mem0 answer, correct answer from PersonaMem, question text, and full question metadata.

**Logs** (in `benchmark_logs/<EXPERIMENT_ID>/`):

- `stage2_qa_<timestamp>.json` — Full log and summary.
- `stage2_qa_<timestamp>_stream.jsonl` — One QA record per line: `retrieved_memories`, `mem0_answer`, `correct_answer`, `user_query`, and all row fields.
- `stage2_qa_<timestamp>_answers.json` — Per-user grouping of QA results.

In [ ]:
from experiment_runner import run_stage2_qa_experiment

# Same experiment_id so we use the memory filled in Chapter 1.
log2 = run_stage2_qa_experiment(
    experiment_id=EXPERIMENT_ID,
    jsonl_path=JSONL_PATH,
    split="benchmark_text",
    max_users=MAX_USERS,
    max_qa_per_user=MAX_QA_PER_USER,
    rerank=True,
    graph_retries=3,
    stream_path=None,
    resume_from_stream=None,
)

print(f"Stage 2 complete: {log2.num_qa_pairs} QA pairs, {log2.total_search_calls} searches")
print(f"Total wall time: {log2.total_wall_seconds:.1f}s")
print(f"Answer LLM calls: {log2.total_answer_llm_calls}")
if log2.total_answer_input_tokens:
    print(f"Answer input tokens: {log2.total_answer_input_tokens}")
if log2.errors:
    print(f"Errors: {len(log2.errors)}")
# Whether graph (Neo4j) was used for retrieval this run (same for all QA)
used_graph_s2 = log2.per_qa[0].get("used_graph") if log2.per_qa else None
print(f"Graph memory: {'enabled' if used_graph_s2 else 'disabled (vector-only fallback)'}")

No sentence-transformers model found with name cross-encoder/ms-marco-MiniLM-L-6-v2. Creating a new one with mean pooling.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1557.05it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
classifier.bias              | UNEXPECTED |  | 
bert.embeddings.position_ids | UNEXPECTED |  | 
classifier.weight            | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Stage 2 (QA): 100%|██████████| 86/86 [08:32<00:00,  5.96s/qa]

Flow log: /Users/viktorzozula/MemoriesTesting/PersonaMem0/benchmark_logs/text_gpt-4.1.-mini_graph/flow_stage2_20260304_195001.log
Stage 2 log written to /Users/viktorzozula/MemoriesTesting/PersonaMem0/benchmark_logs/text_gpt-4.1.-mini_graph/stage2_qa_20260304_195001.json
Stage 2 answers written to /Users/viktorzozula/MemoriesTesting/PersonaMem0/benchmark_logs/text_gpt-4.1.-mini_graph/stage2_qa_20260304_195001_answers.json
Stage 2 complete: 86 QA pairs, 86 searches
Total wall time: 162.8s
Answer LLM calls: 86
Answer input tokens: 32494


In [8]:
# Inspect one QA record: question, retrieved memories, mem0 answer, correct answer
import json
qa = next((q for q in log2.per_qa if q.get("retrieved_memories") and q.get("mem0_answer")), log2.per_qa[0] if log2.per_qa else None)
if qa:
    print("Question (user_query):", qa.get("user_query", "")[:200])
    print("Retrieved memories count:", len(qa.get("retrieved_memories") or []))
    print("Mem0 answer:", (qa.get("mem0_answer") or "")[:300])
    print("Correct answer (PersonaMem):", (qa.get("correct_answer") or "")[:300])
    print("Used graph:", qa.get("used_graph"))

Question (user_query): {'role': 'user', 'content': 'What are some good ways to find fresh, seasonal produce in my area without having to rely on big supermarkets?'}
Retrieved memories count: 10
Mem0 answer: In Pune, you can find fresh, seasonal produce by visiting local farmers' markets or organic produce stalls that often operate on weekends or certain days in residential areas like Kothrud or Baner. Exploring neighborhood weekly bazaars or open-air markets such as the one at Fergusson College Road ca
Correct answer (PersonaMem): In Pune, you might explore your neighborhood’s early morning farmer haats or weekly subzi mandis, where local growers bring freshly harvested, seasonal produce. Building a rapport with a few trusted vendors will help you get the best picks before they sell out, much like maintaining a well-coordinat
Used graph: True


---
# Chapter 3: Running LLM-as-a-Judge

*(To be added later.)*

In [9]:
# Placeholder: LLM-as-a-Judge evaluation will go here.
pass